# Step 3 — Live TomTom traffic

In [9]:
import sys
from IPython.display import display
import pandas as pd

from routing.config import ORIGIN, DESTINATION, MAX_TRAFFIC_REQUESTS
from routing.map_loader import load_road_graph
from routing.map_view import interactive_map
from routing.traffic_api import TomTomClient, TomTomAPIError
from routing.traffic_pipeline import (
    preview_probes,
    collect_traffic_snapshot,
    save_summary,
    make_routing_graph,
)

print("Python kernel:", sys.executable)
G = load_road_graph()
print("OSM nodes:", G.number_of_nodes())
print("Directed edges:", G.number_of_edges())

Python kernel: /Users/mojtabashabani/wiser/mary/.venv/bin/python
OSM nodes: 747
Directed edges: 1718


In [10]:
candidates = preview_probes(G, limit=5)
display(pd.DataFrame(candidates))
print("The preview above made ZERO API calls.")

,edge_id,latitude,longitude,length_m
0,"(1011740925, 1280209951, 0)",51.504716,-0.192008,75.3
1,"(26389034, 95951000, 0)",51.493468,-0.185766,44.4
2,"(26966492, 2180739564, 0)",51.496203,-0.179101,187.9
3,"(414798360, 78553405, 0)",51.501050,-0.186405,92.9
4,"(96087926, 26389322, 0)",51.495261,-0.168308,67.5


The preview above made ZERO API calls.


In [11]:
RUN_ONE_REQUEST = False  # <-- True = ONE billed / quota-counted request

if RUN_ONE_REQUEST:
    candidate = preview_probes(G, limit=1)[0]
    try:
        with TomTomClient() as client:
            response = client.fetch_flow(candidate["latitude"], candidate["longitude"])
        print({k: response.get(k) for k in (
            "currentSpeed", "freeFlowSpeed", "confidence", "roadClosure", "frc"
        )})
        print("OSM probe edge:", candidate["edge_id"])
        print("Reminder: API success alone does NOT prove the OSM edge matches.")
    except TomTomAPIError as err:
        print("TomTom error:", err)
else:
    print("Skipped: no live request sent.")

Skipped: no live request sent.


In [12]:
RUN_BATCH = False            # <-- Change to True only when ready
REQUEST_LIMIT = 12           # Start small. Increase only after inspecting results.

snapshot = None
summary = None

if RUN_BATCH:
    with TomTomClient() as client:
        snapshot, summary = collect_traffic_snapshot(
            G,
            client,
            max_requests=REQUEST_LIMIT,
        )
    save_summary(summary)  
    print("Requests:", summary["requests_made"])
    print("Collection UTC:", summary["started_at_utc"], "to", summary["finished_at_utc"])
    print("Matched candidates:", summary["geometry_and_heading_candidate_matches"])
    print("Usable traffic:", summary["usable_observed_edges"])
    print("Coverage:", summary["usable_coverage_percent"], "%")
    print("Statuses:", summary["counts"])
    if summary["stopped_http_status"]:
        print("Stopped early due to HTTP", summary["stopped_http_status"])
else:
    print("Skipped batch: zero HTTP requests.")

Skipped batch: zero HTTP requests.


In [13]:
if snapshot is not None:
    display(pd.DataFrame([
        {"status": status, "edges": count}
        for status, count in summary["counts"].items()
    ]))
    print("Missing and other reasons:", summary["reasons"])
    rows = []
    for u, v, key, e in snapshot.edges(keys=True, data=True):
        if e["traffic_status"] in ("red", "yellow", "clear", "closed"):
            rows.append({
                "edge_id": (u, v, key),
                "status": e["traffic_status"],
                "length_m": round(float(e["length"]), 1),
                "current_kph": e["current_speed_kph"],
                "free_kph": e["free_flow_speed_kph"],
                "travel_seconds": e["travel_time_s"],
                "confidence": e["traffic_confidence"],
            })
    display(pd.DataFrame(rows[:12]))  # Live data ONLY in this notebook session
else:
    print("Run the collection cell first.")

Run the collection cell first.


In [14]:
if snapshot is not None:
    free_time_graph = make_routing_graph(snapshot, objective="time", scenario="free_flow")
    measured_time_graph = make_routing_graph(snapshot, objective="time", scenario="measured")
    measured_distance_graph = make_routing_graph(snapshot, objective="distance", scenario="measured")

    extra_closed_edge = next((
        (u, v, k) for u, v, k, e in snapshot.edges(keys=True, data=True)
        if not e["is_closed"]
    ), None)
    if extra_closed_edge is not None:
        closure_time_graph = make_routing_graph(
            snapshot, objective="time", scenario="closure",
            imposed_closure=extra_closed_edge,
        )
        print("Example imposed directed closure:", extra_closed_edge)
        print("Closure scenario weight:", closure_time_graph.graph["routing_weight"])
        print("Edge removed:", not closure_time_graph.has_edge(*extra_closed_edge))
    print("Free-flow weight:", free_time_graph.graph["routing_weight"])
    print("Measured weight:", measured_time_graph.graph["routing_weight"])
    print("Distance weight:", measured_distance_graph.graph["routing_weight"])
else:
    print("Collect one snapshot first.")

Collect one snapshot first.


In [15]:
RUN_LIVE_TRAFFIC = True

if RUN_LIVE_TRAFFIC:
    client = TomTomClient()
    try:
        snapshot, summary = collect_traffic_snapshot(
            G, client, max_requests=MAX_TRAFFIC_REQUESTS
        )
    finally:
        client.close()
    save_summary(summary)
    print(summary)
else:
    print("No TomTom requests made. Set RUN_LIVE_TRAFFIC=True when ready.")

{'started_at_utc': '2026-10-25T11:06:40.266362+00:00', 'finished_at_utc': '2026-10-25T11:06:56.269130+00:00', 'requests_made': 60, 'request_errors': 14, 'stopped_http_status': None, 'total_directed_edges': 1718, 'sampled_directed_edges': 60, 'geometry_and_heading_candidate_matches': 16, 'usable_observed_edges': 16, 'matched_coverage_percent': 0.93, 'usable_coverage_percent': 0.93, 'counts': {'missing': 1702, 'red': 0, 'yellow': 3, 'clear': 12, 'closed': 1}, 'reasons': {'not_queried': 1658, 'observed': 15, 'reported_closure': 1, 'geometry_or_heading_mismatch': 30, 'api_http_400': 14}, 'fallback_speed_kph': 30.0, 'minimum_confidence': 0.65, 'heading_policy': 'TomTom coordinate-order proxy; not authoritative direction verification', 'notes': 'Unqueried/unmatched/low-confidence edges are missing, never clear. Imputed costs are not observations.'}


In [16]:
if RUN_LIVE_TRAFFIC:
    traffic_widget, state = interactive_map(G, ORIGIN, DESTINATION, traffic_graph=snapshot)
    display(traffic_widget)

    time_graph = make_routing_graph(snapshot, objective="time")
    distance_graph = make_routing_graph(snapshot, objective="distance")
    print("Traffic-weighted route-ready edges:", time_graph.number_of_edges())

Traffic-weighted route-ready edges: 1717
